# 05. Llama 3.2 predictions for dataset B

Fills the `tool_predicted` column for the 860 new rows of dataset B, using the same local
model and setup as the original collection run on dataset A:

- Model: **Llama 3.2 3B**, run locally with Ollama (`llama3.2:3b`).
- Temperature 0, seed 42.
- The model gets the list of tools for the query's role and must answer with one tool name.

Dataset A's stored predictions are kept as they are. A short consistency check at the start
re-runs 30 rows of A to confirm this notebook reproduces them.

**Replication without Ollama:** the predictions are saved to `llama_predictions_B.csv`. If
Ollama is not running and that file exists, the notebook loads it instead of calling the model.

In [1]:
import os, socket
import pandas as pd

B = pd.read_csv("cyber_dataset_B.csv")
CACHE = "llama_predictions_B.csv"
MODEL = "llama3.2:3b"

# same tool lists, in the same order, as the original collection run
ROLE_TOOLS = {
    "attacker": ["SSHConnect", "NmapScan", "PortScan", "CheckVulnerability"],
    "defender": ["ReadAuthLog", "ListeningPorts", "BlockIP", "CheckFailedLogins", "ListProcesses"],
    "shared":   ["GetSystemInfo", "ReadSyslog"],
}
ROLE_INTRO = {"attacker": "a penetration-testing agent",
              "defender": "a defensive security agent",
              "shared":   "a system-inspection agent"}

def system_prompt(role):
    return (f"You are a tool selector for {ROLE_INTRO[role]}. Given a task, "
            f"respond with exactly one word \u2014 the tool to use, chosen from: "
            f"{', '.join(ROLE_TOOLS[role])}.")

def ollama_up():
    s = socket.socket(); s.settimeout(2)
    ok = s.connect_ex(("127.0.0.1", 11434)) == 0
    s.close()
    return ok

def pick_tool(prompt, role):
    import ollama
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "system", "content": system_prompt(role)},
                              {"role": "user", "content": prompt}],
                    options={"temperature": 0, "seed": 42})
    raw = r["message"]["content"].strip()
    valid = set(ROLE_TOOLS[role])
    for tok in raw.replace(".", " ").replace(":", " ").replace(",", " ").split():
        if tok in valid:
            return tok
    return "Unknown"

print("rows:", len(B), "| rows still missing tool_predicted:", B["tool_predicted"].isna().sum())
print("Ollama running:", ollama_up())

rows: 1500 | rows still missing tool_predicted: 860
Ollama running: True


## 1. Consistency check on dataset A

Re-run 30 rows of A and compare with the stored predictions. Small differences are possible
because model output is not guaranteed identical across runs and machines.

In [2]:
if ollama_up():
    A = B[B["dataset"] == "A"].sample(30, random_state=0)
    again = [pick_tool(p, r) for p, r in zip(A["prompt"], A["agent_role"])]
    same = sum(a == b for a, b in zip(again, A["tool_predicted"]))
    print(f"re-run matches the stored prediction on {same}/30 rows of dataset A")
else:
    print("Ollama not running: skipping the consistency check")

re-run matches the stored prediction on 30/30 rows of dataset A


## 2. Predict the new rows

In [3]:
new_mask = B["dataset"] == "B"      # the 860 new rows (safe to re-run)
if os.path.exists(CACHE) and not ollama_up():
    cache = pd.read_csv(CACHE)
    print("Ollama not running: using", CACHE)
elif ollama_up():
    rows = B[new_mask]
    preds = []
    for i, (p, r) in enumerate(zip(rows["prompt"], rows["agent_role"]), 1):
        preds.append(pick_tool(p, r))
        if i % 100 == 0:
            print(f"  {i}/{len(rows)}")
    cache = pd.DataFrame({"prompt": rows["prompt"].values, "tool_predicted": preds})
    cache.to_csv(CACHE, index=False)
    print("saved", CACHE, "with", len(cache), "predictions")
else:
    raise SystemExit("Ollama is not running and there is no cached predictions file.")

lookup = dict(zip(cache["prompt"], cache["tool_predicted"]))
B.loc[new_mask, "tool_predicted"] = B.loc[new_mask, "prompt"].map(lookup)
assert B["tool_predicted"].notna().all()
print("all", len(B), "rows now have a tool_predicted")

  100/860


  200/860


  300/860


  400/860


  500/860


  600/860


  700/860


  800/860


saved llama_predictions_B.csv with 860 predictions
all 1500 rows now have a tool_predicted


## 3. Check the predictions

How often the model answered with something that is not one of its tools (`Unknown`), and how
accurate it is on dataset A versus the new rows. This is only a look at the numbers; the
full comparison with the trained classifier is in notebook 06.

In [4]:
B["llama_correct"] = B["tool_predicted"] == B["tool_ground_truth"]
print("Unknown answers:", int((B["tool_predicted"] == "Unknown").sum()))
print()
print("Llama 3.2 accuracy:")
print(f"  dataset A (640)      : {B[B.dataset == 'A']['llama_correct'].mean():.1%}")
print(f"  new rows only (860)  : {B[B.dataset == 'B']['llama_correct'].mean():.1%}")
print(f"  dataset B (all 1500) : {B['llama_correct'].mean():.1%}")
print()
print("by category (new rows):")
print(B[B.dataset == "B"].groupby("category")["llama_correct"].agg(n="size", acc="mean")
      .assign(acc=lambda d: (d.acc * 100).round(1)).to_string())

Unknown answers: 5

Llama 3.2 accuracy:
  dataset A (640)      : 79.4%
  new rows only (860)  : 76.0%
  dataset B (all 1500) : 77.5%

by category (new rows):
                n   acc
category               
cross_role     76  73.7
messy         176  76.7
near_miss     173  71.1
no_cue        171  73.1
shared_vocab  177  80.8
vocab_shift    87  82.8


## 4. Save

In [5]:
B.drop(columns=["llama_correct"]).to_csv("cyber_dataset_B.csv", index=False)
print("saved cyber_dataset_B.csv with tool_predicted filled for all", len(B), "rows")

saved cyber_dataset_B.csv with tool_predicted filled for all 1500 rows
